<span style="font-size:2.5em">8. 순환 신경망</span>

이 자료는 다음 자료들을 기반으로 일부 수정된 자료입니다.<br>
- [유원준, 안상준, "PyTorch로 시작하는 딥 러닝 입문"](https://wikidocs.net/60760)

# 8.1 순환 신경망(Recurrent Neural Network, RNN)
## 1) Vanilla RNN
* 시퀀스 모델로서 입력과 출력을 시퀀스 단위로 처리하는 모델
* 셀(cell) : 은닉층에서 활성화 함수를 통해 결과를 내보내는 역할 / 이전값을 기억하려는 메모리 역할을 수행함 (Vanilla RNN, LSTM, GRU 등 은닉층의 기능을 강조하는 포괄적 명칭).
<img src="images/rnn_image4_ver2.png" width="100" style="margin-left: auto; margin-right: auto">
<p style="text-align: center;">Vanilla RNN 구조</p>

$𝐡^{(𝑡)}=\tanh⁡(𝐖_𝑥 𝐱^{(𝑡)}+𝐖_ℎ 𝐡^{(𝑡−1)}+𝐛)$ <br>
$𝐲^{(𝑡)}=\text{softmax}(𝐖_𝑦 𝐡^{(𝑡)})$

### Python으로 RNN 구현하기

In [1]:
import numpy as np

np.random.seed(0)

timesteps = 10  # 시점의 수 또는 시퀀스의 길이
input_size = 4  # 입력의 차원, NLP에서는 단어 벡터의 차원
hidden_size = 8 # 은닉 상태의 크기(메모리 셀의 용량)

inputs = np.random.random((timesteps, input_size))  # 입력시퀀스 크기 정의 (random으로 초기화)
hidden_state_t = np.zeros((hidden_size,))           # 은닉상태 크기 정의 (0으로 초기화)

- inputs 의 차원 = timesteps x input_size
- hidden_state_t의 차원  = hidden_size 

그러나, 실제로는 minibatch(크기 = batch_size)를 사용하므로,
- inputs 의 차원 = batch_size x timesteps x input_size
- hidden_state_t의 차원  = batch_size x hidden_size

In [2]:
Wx = np.random.random((hidden_size, input_size))  # (8, 4)크기의 2D 텐서 생성. 입력에 대한 가중치.
Wh = np.random.random((hidden_size, hidden_size)) # (8, 8)크기의 2D 텐서 생성. 은닉 상태에 대한 가중치.
b = np.random.random((hidden_size,))              # (8, ) bias

In [3]:
print(np.shape(Wx))
print(np.shape(Wh))
print(np.shape(b))

(8, 4)
(8, 8)
(8,)


 모든 time step에서 hidden_state를 출력한다고 가정한다.

In [4]:
total_hidden_states = []  # 모든 time_step의 hidden_state의 저장소

for input_t in inputs:
    output_t = np.tanh(np.dot(Wx, input_t) + np.dot(Wh, hidden_state_t) + b)
    total_hidden_states.append(list(output_t))
    hidden_state_t = output_t

total_hidden_states = np.stack(total_hidden_states, axis = 0)
print(total_hidden_states)

[[0.91170118 0.93488513 0.93128902 0.85123792 0.95178278 0.76354041
  0.87528299 0.97864588]
 [0.99961839 0.99954504 0.99991363 0.99989286 0.99997858 0.99997573
  0.99989965 0.99995934]
 [0.99987737 0.9998427  0.9999721  0.99998009 0.99999614 0.99999447
  0.99997501 0.99999388]
 [0.9997078  0.99980461 0.9999192  0.99993932 0.99998807 0.99998382
  0.99993042 0.99993387]
 [0.99984047 0.99972104 0.99997255 0.99989437 0.99999098 0.99999374
  0.99997513 0.99997965]
 [0.99987036 0.99990496 0.99997615 0.99998122 0.99999632 0.99999278
  0.99997007 0.99999171]
 [0.9995775  0.99959404 0.99994268 0.99988487 0.99998475 0.99998573
  0.99992293 0.9999423 ]
 [0.9996679  0.99968142 0.999947   0.99994544 0.9999902  0.9999882
  0.99993416 0.99997161]
 [0.99956312 0.99967339 0.99992502 0.99992985 0.99998659 0.99998321
  0.99990694 0.99994243]
 [0.99989776 0.99983429 0.9999784  0.99996602 0.99999563 0.99999539
  0.99998173 0.99999321]]


### PyTorch의 nn.RNN()

In [5]:
import torch
import torch.nn as nn
torch.manual_seed(0)

In [6]:
batch_size = 1
time_steps = 10
input_size = 5
hidden_size = 8

#### 입력 텐서의 정의
- 입력텐서의 크기 = (batch_size, time_steps, input_size)

In [7]:
inputs = torch.Tensor(batch_size, time_steps, input_size)
inputs

tensor([[[5.7680e+17, 1.0916e-42, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00]]])

In [9]:
cell = nn.RNN(input_size, hidden_size, batch_first=True)

* `batch_first=True`: 입력 텐서의 첫번째 차원이 `batch_size`임을 알려줌

In [10]:
outputs, _status = cell(inputs)

In [11]:
print(outputs.shape, _status.shape) 

torch.Size([1, 10, 8]) torch.Size([1, 1, 8])


In [12]:
print(outputs) # 모든 time-step의 hidden_state
print(_status) # 마지막 time-step의 hidden_state

tensor([[[-1.0000,  1.0000, -1.0000,  1.0000,  1.0000,  1.0000, -1.0000,
          -1.0000],
         [-0.9647,  0.6310,  0.1029,  0.2162,  0.0024,  0.3075,  0.5192,
           0.7922],
         [-0.4824,  0.4855, -0.2960, -0.7194, -0.2073,  0.3452, -0.5557,
           0.0212],
         [-0.5070, -0.0844,  0.1486, -0.4844, -0.4552,  0.4999, -0.2399,
           0.5151],
         [-0.2903,  0.0542,  0.1208, -0.5613, -0.5480,  0.2831, -0.2868,
           0.2112],
         [-0.2972,  0.0140,  0.0501, -0.5698, -0.4231,  0.3288, -0.2711,
           0.2336],
         [-0.3210,  0.0154,  0.1048, -0.5461, -0.4608,  0.3061, -0.2489,
           0.2941],
         [-0.2975,  0.0261,  0.0779, -0.5680, -0.4505,  0.3082, -0.2735,
           0.2470],
         [-0.3103,  0.0140,  0.0891, -0.5559, -0.4498,  0.3134, -0.2588,
           0.2738],
         [-0.3046,  0.0212,  0.0873, -0.5603, -0.4538,  0.3077, -0.2644,
           0.2628]]], grad_fn=<TransposeBackward1>)
tensor([[[-0.3046,  0.0212,  0.0873, -

* hidden layer 수가 1 이므로, 당연히 outputs의 마지막 벡터는 \_status와 동일하다. 

### 심층 순환신경망(Deep RNN)
<img src="images/rnn_image4.5_finalPNG.png" width="200" style="margin-left: auto; margin-right: auto">
<p style="text-align: center;">Deep RNN 구조</p>

In [13]:
inputs = torch.Tensor(batch_size, time_steps, input_size)

In [14]:
cell = nn.RNN(input_size, hidden_size, num_layers=2, batch_first=True)

In [15]:
outputs, _status = cell(inputs)

In [17]:
print(outputs.shape, _status.shape)

torch.Size([1, 10, 8]) torch.Size([2, 1, 8])


In [18]:
_status

tensor([[[-0.1486,  0.5987, -0.0590, -0.2331,  0.1348, -0.0103, -0.0988,
           0.3451]],

        [[ 0.2919,  0.1920,  0.1734,  0.3562,  0.4687, -0.4474, -0.2594,
          -0.1932]]], grad_fn=<StackBackward0>)

In [17]:
outputs

tensor([[[ 0.5190,  0.8253, -0.7381,  0.8584,  0.5571, -0.2272,  0.2554,
           0.4334],
         [ 0.3579,  0.2998,  0.3992,  0.4562,  0.7477, -0.5177, -0.1954,
           0.1010],
         [ 0.3321,  0.1758,  0.0666,  0.3272,  0.5856, -0.4590, -0.3007,
          -0.2223],
         [ 0.3205,  0.1124,  0.2037,  0.3697,  0.4317, -0.5654, -0.3246,
          -0.1320],
         [ 0.3366,  0.1958,  0.1003,  0.3160,  0.4774, -0.4814, -0.2820,
          -0.2530],
         [ 0.2987,  0.1653,  0.1783,  0.3654,  0.4457, -0.4956, -0.2843,
          -0.1935],
         [ 0.3081,  0.1998,  0.1428,  0.3357,  0.4694, -0.4512, -0.2624,
          -0.2117],
         [ 0.2915,  0.1861,  0.1756,  0.3606,  0.4621, -0.4573, -0.2644,
          -0.1936],
         [ 0.2981,  0.1983,  0.1619,  0.3458,  0.4712, -0.4435, -0.2572,
          -0.1964],
         [ 0.2919,  0.1920,  0.1734,  0.3562,  0.4687, -0.4474, -0.2594,
          -0.1932]]], grad_fn=<TransposeBackward1>)

- _status[1]과 outputs[0,9]의 값이 같다.

### 양방향 순환신경망(Bidirectional RNN)
#### Bidirectional RNN (1  layer)
<img src="images/rnn_image5_ver2.png" width="300" style="margin-left: auto; margin-right: auto">

#### Bidirectional RNN (2  layer)
<img src="images/rnn_image6_ver3.png" width="300" style="margin-left: auto; margin-right: auto">

In [19]:
inputs = torch.Tensor(batch_size, time_steps, input_size)  # (1, 10, 5)

In [20]:
cell = nn.RNN(input_size = 5, hidden_size = 8, num_layers = 2, batch_first=True, bidirectional = True)

In [21]:
outputs, _status = cell(inputs)

In [23]:
print(outputs.shape, _status.shape)   # hidden_size 위치가 8 --> 16 (양방향을 concat)
                                      # _status : (1, 1, 8) --> (4, 1, 8) : 2 layer x bidirection 

torch.Size([1, 10, 16]) torch.Size([4, 1, 8])


## 2) Long Short-Term Memory(LSTM)
* 바닐라RNN은 짧은 시퀀스에 대해서만 효과를 보임
* timestamp가 길어질수록 앞의 정보가 전달되지 못함
* LSTM은 cell state라는 값을 추가함 (은닉 상태값과 셀 상태값을 구하기 위해서 새로 추가 된 3개의 게이트를 사용)
<img src="images/LSTM_cell_01.png" width="300" style="margin-left: auto; margin-right: auto">
<p style="text-align: center;">LSTM Cell 구조</p>
#### (1) Forget gate
* 삭제 게이트는 기억을 삭제하기 위한 게이트 <br>
$𝐟^{(𝑡)}=𝛔(𝐛_𝑓+𝐔_𝑓 𝐱^{(𝑡)}+𝐖_𝑓 𝐡^{(𝑡-1)})$
#### (2) Input gate
* 입력게이트는 현재 정보를 기억하기 위한 게이트 <br>
$𝐢^{(𝑡)}=𝛔(𝐛_𝑖+𝐔_i 𝐱^{(𝑡)}+𝐖_h𝑖 𝐡^{(𝑡-1)})$ <br>
$𝐠^{(𝑡)}=\tanh(𝐛+𝐔𝐱^{(𝑡)}+𝐖𝐡^{(𝑡-1)})$
#### (3) Update cell states  (장기 상태)
* 셀 상태 Ct를 LSTM에서는 장기 상태라고 부르기도 함 <br> 
$𝐜^{(𝑡)}=𝐟^{(𝑡)}⊙𝐜^{(𝑡-1)}+𝐢^{(𝑡)}⊙𝐠^{(𝑡)}$ <br>
#### (4) Output gate (단기 상태)
* 단기 상태의 값은 또한 출력층으로도 향함 <br> 
$𝐪^{(𝑡)}=𝛔(𝐛_𝑧+𝐔_𝑧 𝐱^{(𝑡)}+𝐖_𝑧 𝐡^{(𝑡-1)})$ <br>
$𝐡^{(𝑡)}=𝐪^{(𝑡)}⊙\tanh(𝐜^{(𝑡)})$

In [24]:
import torch
import torch.nn as nn

LSTM 셀을 사용하는 것은 vanilla RNN과 거의 같다. 

In [25]:
nn.RNN(input_size, hidden_size, batch_first=True)

RNN(5, 8, batch_first=True)

In [26]:
nn.LSTM(input_size, hidden_size, batch_first=True)

LSTM(5, 8, batch_first=True)

## 3) GRU(Gated Recurrent Units)
* Forget Gate와 Input Gate를 하나로 묶어 Update Gate를 만듬
* Cell State 와 Hidden State 를 통합함 <br>
$𝐳^{(𝑡)}=𝛔(𝐛_𝑧+𝐔_𝑧 𝐱^{(𝑡)}+𝐖_𝑧 𝐡^{(𝑡-1)})$ <br>
$𝐫^{(𝑡)}=𝛔(𝐛_𝑟+𝐔_𝑟 𝐱^{(𝑡)}+𝐖_𝑟 𝐡^{(𝑡-1)})$ <br>
$𝐡 ̃^{(𝑡)}=\tanh{𝐛+𝐔𝐱^{(𝑡)}+𝐖(𝐫^{(𝑡)}⊙𝐡^{(𝑡-1)})}$ <br>
$𝐡^{(𝑡)}=(1−𝐳^{(𝑡)})⊙𝐡^{(𝑡-1)}+𝐳^{(𝑡)}⊙𝐡 ̃^{(𝑡)}$

GRU 셀을 사용하는 것은 vanilla RNN과 거의 같다. 

In [27]:
nn.RNN(input_size, hidden_size, batch_first=True)

RNN(5, 8, batch_first=True)

In [28]:
nn.GRU(input_size, hidden_size, batch_first=True)

GRU(5, 8, batch_first=True)

# 8.2 문자 단위 RNN (Char RNN)
* 다대다 RNN은 대표적으로 품사태깅, 개체명인식 등에서 사용됨
## 1) Char RNN - 작은 데이터로 실험

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
np.random.seed(0)
torch.manual_seed(0)

In [30]:
input_str = 'apple'
label_str = 'pple!'

In [31]:
char_vocab = sorted(list(set(input_str+label_str)))
char_vocab

['!', 'a', 'e', 'l', 'p']

In [32]:
vocab_size = len(char_vocab)
print("문자 집합의 크기 : {}".format(vocab_size))

문자 집합의 크기 : 5


In [33]:
input_size = vocab_size # 입력의 크기는 문자 집합의 크기
hidden_size = 5
output_size = 5
learning_rate = 0.1

In [34]:
char_to_index = {char: i for i, char in enumerate(char_vocab)}
char_to_index

{'!': 0, 'a': 1, 'e': 2, 'l': 3, 'p': 4}

In [59]:
index_to_char = {}
for key, value in char_to_index.items():
    index_to_char[value] = key
print(index_to_char)

{0: '!', 1: 'a', 2: 'e', 3: 'l', 4: 'p'}


In [60]:
x = [char_to_index[char] for char in input_str]
y = [char_to_index[char] for char in label_str]
print(x)
print(y)

[1, 4, 4, 3, 2]
[4, 4, 3, 2, 0]


In [61]:
# batch 차원 추가
x = [x]
y = [y]
print(x)
print(y)

[[1, 4, 4, 3, 2]]
[[4, 4, 3, 2, 0]]


In [62]:
x_one_hot = [np.eye(vocab_size)[x_data] for x_data in x]
print(x_one_hot)

[array([[0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.]])]


In [63]:
X = torch.FloatTensor(np.array(x_one_hot))
Y = torch.LongTensor(y)

In [64]:
print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))

훈련 데이터의 크기 : torch.Size([1, 5, 5])
레이블의 크기 : torch.Size([1, 5])


In [65]:
X

tensor([[[0., 1., 0., 0., 0.],
         [0., 0., 0., 0., 1.],
         [0., 0., 0., 0., 1.],
         [0., 0., 0., 1., 0.],
         [0., 0., 1., 0., 0.]]])

In [66]:
Y

tensor([[4, 4, 3, 2, 0]])

In [67]:
class Net(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(Net, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size, bias=True)

    def forward(self, x):
        net, _status = self.rnn(x)
        net = self.fc(net)
        return net

In [68]:
net = Net(input_size, hidden_size, output_size)

In [69]:
outputs = net(X)
print(outputs.shape)  # (batch_size, time_steps, out_size) = (1, 5, 5)

torch.Size([1, 5, 5])


In [70]:
outputs

tensor([[[-0.4130, -0.0243, -0.1370, -0.2147,  0.3572],
         [-0.2985, -0.0466, -0.0074, -0.1192,  0.2490],
         [-0.2811, -0.1210, -0.0304, -0.1537,  0.2217],
         [-0.2289,  0.0118,  0.0395, -0.0031,  0.1139],
         [-0.1409, -0.3234, -0.0750, -0.5121,  0.1957]]],
       grad_fn=<ViewBackward0>)

In [71]:
print(outputs.view(-1, input_size).shape)

torch.Size([5, 5])


In [72]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([1, 5])
torch.Size([5])


In [73]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [74]:
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X)
    loss = loss_fn(outputs.view(-1,input_size), Y.view(-1))

    loss.backward()
    optimizer.step()

    result = outputs.data.numpy().argmax(axis=2)
    result_str = ''.join([index_to_char[c] for c in np.squeeze(result)])
    print(f"{i} loss: {loss.item()} prediction: {result} true Y: {y} prediction str: {result_str}")

0 loss: 1.4814866781234741 prediction: [[4 4 4 4 4]] true Y: [[4, 4, 3, 2, 0]] prediction str: ppppp
1 loss: 1.2011629343032837 prediction: [[4 4 4 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pppe!
2 loss: 0.9456263780593872 prediction: [[4 4 3 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pple!
3 loss: 0.6897298693656921 prediction: [[4 4 3 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pple!
4 loss: 0.48517942428588867 prediction: [[4 4 3 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pple!
5 loss: 0.3391508162021637 prediction: [[4 4 3 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pple!
6 loss: 0.23791077733039856 prediction: [[4 4 3 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pple!
7 loss: 0.16768835484981537 prediction: [[4 4 3 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pple!
8 loss: 0.11902425438165665 prediction: [[4 4 3 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pple!
9 loss: 0.08464252203702927 prediction: [[4 4 3 2 0]] true Y: [[4, 4, 3, 2, 0]] predict

## 2) Char RNN - 더 많은 데이터

In [75]:
import torch
import torch.nn as nn
import torch.optim as optim
np.random.seed(0)
torch.manual_seed(0)

In [76]:
sentence = ("if you want to build a ship, don't drum up people together to "
            "collect wood and don't assign them tasks and work, but rather "
            "teach them to long for the endless immensity of the sea.")

In [77]:
len(sentence)

180

In [78]:
char_set = list(set(sentence))

In [79]:
print(char_set)

["'", 'k', '.', 'n', 'm', 'g', 'w', 'u', ' ', 'c', 'd', 'a', 'f', 'r', 'o', ',', 'y', 'e', 'p', 'i', 's', 'b', 't', 'h', 'l']


In [80]:
char_dic = {char: i for i, char in enumerate(char_set)}
print(char_dic)

{"'": 0, 'k': 1, '.': 2, 'n': 3, 'm': 4, 'g': 5, 'w': 6, 'u': 7, ' ': 8, 'c': 9, 'd': 10, 'a': 11, 'f': 12, 'r': 13, 'o': 14, ',': 15, 'y': 16, 'e': 17, 'p': 18, 'i': 19, 's': 20, 'b': 21, 't': 22, 'h': 23, 'l': 24}


In [81]:
dic_size = len(char_dic)
print('문자 집합의 크기 : {}'.format(dic_size))

문자 집합의 크기 : 25


In [82]:
hidden_size = dic_size
sequence_length = 10  # 임의 숫자 지정
learning_rate = 0.1

In [83]:
x_data = []
y_data = []

for i in range(0, len(sentence) - sequence_length):
    x_str = sentence[i : i + sequence_length]
    y_str = sentence[i+1 : i+1+ sequence_length]
    print(i, x_str, "->", y_str)

    x_data.append([char_dic[char] for char in x_str])
    y_data.append([char_dic[char] for char in y_str])

0 if you wan -> f you want
1 f you want ->  you want 
2  you want  -> you want t
3 you want t -> ou want to
4 ou want to -> u want to 
5 u want to  ->  want to b
6  want to b -> want to bu
7 want to bu -> ant to bui
8 ant to bui -> nt to buil
9 nt to buil -> t to build
10 t to build ->  to build 
11  to build  -> to build a
12 to build a -> o build a 
13 o build a  ->  build a s
14  build a s -> build a sh
15 build a sh -> uild a shi
16 uild a shi -> ild a ship
17 ild a ship -> ld a ship,
18 ld a ship, -> d a ship, 
19 d a ship,  ->  a ship, d
20  a ship, d -> a ship, do
21 a ship, do ->  ship, don
22  ship, don -> ship, don'
23 ship, don' -> hip, don't
24 hip, don't -> ip, don't 
25 ip, don't  -> p, don't d
26 p, don't d -> , don't dr
27 , don't dr ->  don't dru
28  don't dru -> don't drum
29 don't drum -> on't drum 
30 on't drum  -> n't drum u
31 n't drum u -> 't drum up
32 't drum up -> t drum up 
33 t drum up  ->  drum up p
34  drum up p -> drum up pe
35 drum up pe -> rum up peo
36

In [84]:
print(x_data[0])  # if you wan에 해당됨.
print(y_data[0])  # f you want에 해당됨.

[19, 12, 8, 16, 14, 7, 8, 6, 11, 3]
[12, 8, 16, 14, 7, 8, 6, 11, 3, 22]


In [86]:
x_one_hot = [np.eye(dic_size)[x] for x in x_data] # x 데이터는 원-핫 인코딩
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)

In [87]:
print(f'훈련 데이터의 크기 : {X.shape}')
print(f'레이블의 크기 : {Y.shape}')

훈련 데이터의 크기 : torch.Size([170, 10, 25])
레이블의 크기 : torch.Size([170, 10])


In [88]:
print(X[0])

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,

In [89]:
print(Y[0])

tensor([12,  8, 16, 14,  7,  8,  6, 11,  3, 22])


In [90]:
class Net(nn.Module):
    def __init__(self, input_dim, hidden_dim, layers):
        super(Net, self).__init__()
        self.rnn = nn.RNN(input_dim, hidden_dim, num_layers=layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, hidden_dim, bias=True)
  
    def forward(self, x):
        net, _status = self.rnn(x)
        net = self.fc(net)
        return net

In [91]:
net = Net(dic_size, hidden_size, 2)

In [92]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [93]:
outputs = net(X)
print(outputs.shape)

torch.Size([170, 10, 25])


In [95]:
print(outputs.view(-1, dic_size).shape) # 2차원 텐서로 변환.

torch.Size([1700, 25])


In [96]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([170, 10])
torch.Size([1700])


In [97]:
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X)

    loss = loss_fn(outputs.view(-1, dic_size), Y.view(-1))
    loss.backward()
    optimizer.step()

    results = outputs.argmax(dim=2)
    predict_str = ""
    for j, result in enumerate(results):
        if j == 0: # 처음에는 예측 결과를 전부 가져오지만
            predict_str += ''.join([char_set[t] for t in result])
        else: # 그 다음에는 마지막 글자만 반복 추가
            predict_str += char_set[result[-1]]

    print(f"{i}  {predict_str}")

0  nrrffurfrrfrfrrffnurrrrurprfrrrururrruurfrurfruuurfrffurfrrurruruurufrfrrrrrrurrrrrfrrfupfrrfrfrrfrffurrrfrfrprfufufrrrurfrrufufrrfrfrrfrrurrfrurruurfrfuruuufrrprruulfffrrrfrfrufr
1  taaaaaaataatataaaaaaaaaaaaaaaaaaaaaaaaaaataaaaaaaaataaaaaatataataaaaaaataaaaaaaaaaaaaaaaaaaaataaaataaaaaaaaatataaaaaaatataataaaataataaaataaaaatatatataaaaaaaaaaaaaaaaaaaaaaaataaaaa
2  totototototo otoototottottootototoootototttoottootooototototottototottotototototttoottootototottoootottotototototototototototottotototototototootototototototototooott ootototototo
3    b.  e..nlnhelhlenenllneelnlneneelnenleeleneelene.tneelneneeleel enelnell.eeeneellnenlnleeteneeehllneltlne ehenenlne lneneeenleel lneeeelnelllneelhenenenlne.lne l.lee nlllnene.ln
4    eh  ehl                                                                                                                                                                          
5  t e n       a aa       a  a aa a   a    a  a aaaa a a                 a  aa   a   

## 3) Word 단위 RNN - 단어 임베딩 사용

In [99]:
import torch
import torch.nn as nn
import torch.optim as optim
np.random.seed(0)
torch.manual_seed(0)

In [100]:
sentence = "Repeat is the best medicine for memory".split()

In [101]:
vocab = list(set(sentence))
print(vocab)

['for', 'is', 'medicine', 'memory', 'the', 'Repeat', 'best']


In [102]:
word2index = {word: i+1 for i, word in enumerate(vocab)}
word2index['<unk'] = 0

In [103]:
print(word2index)

{'for': 1, 'is': 2, 'medicine': 3, 'memory': 4, 'the': 5, 'Repeat': 6, 'best': 7, '<unk': 0}


In [104]:
print(word2index['memory'])

4


In [105]:
index2word = {value: key for key,value in word2index.items()}
print(index2word)

{1: 'for', 2: 'is', 3: 'medicine', 4: 'memory', 5: 'the', 6: 'Repeat', 7: 'best', 0: '<unk'}


In [106]:
print(index2word[7])

best


In [107]:
def build_data(sentence, word2index):
    encoded = [word2index[word] for word in sentence]
    input_seq, label_seq = encoded[:-1], encoded[1:]

    input_seq = torch.LongTensor(input_seq).unsqueeze(0) # 배치 차원 추가
    label_seq = torch.LongTensor(label_seq).unsqueeze(0) # 배치 차원 추가
    return input_seq, label_seq

In [108]:
X, Y = build_data(sentence, word2index)

In [109]:
print(X)
print(Y)

tensor([[6, 2, 5, 7, 3, 1]])
tensor([[2, 5, 7, 3, 1, 4]])


In [110]:
class Net(nn.Module):
    def __init__(self, vocab_size, input_size, hidden_size, batch_first=True):
        super(Net, self).__init__()
        self.embedding_layer = nn.Embedding(num_embeddings=vocab_size, # 워드 임베딩
                                            embedding_dim=input_size)
        self.rnn_layer = nn.RNN(input_size, hidden_size, # 입력 차원, 은닉 상태의 크기 정의
                                batch_first=batch_first)
        self.linear = nn.Linear(hidden_size, vocab_size) # 출력은 원-핫 벡터의 크기를 가져야함. 또는 단어 집합의 크기만큼 가져야함.

    def forward(self, x):
        # 1. 임베딩 층
        # 크기변화: (배치 크기, 시퀀스 길이) => (배치 크기, 시퀀스 길이, 임베딩 차원)
        output = self.embedding_layer(x)
        # 2. RNN 층
        # 크기변화: (배치 크기, 시퀀스 길이, 임베딩 차원)
        # => output (배치 크기, 시퀀스 길이, 은닉층 크기), hidden (1, 배치 크기, 은닉층 크기)
        output, hidden = self.rnn_layer(output)
        # 3. 최종 출력층
        # 크기변화: (배치 크기, 시퀀스 길이, 은닉층 크기) => (배치 크기, 시퀀스 길이, 단어장 크기)
        output = self.linear(output)
        # 4. view를 통해서 배치 차원 제거
        # 크기변화: (배치 크기, 시퀀스 길이, 단어장 크기) => (배치 크기*시퀀스 길이, 단어장 크기)
        return output.view(-1, output.size(2))

In [111]:
# 하이퍼 파라미터
vocab_size = len(word2index)  # 단어장의 크기는 임베딩 층, 최종 출력층에 사용된다. <unk> 토큰을 크기에 포함한다.
input_size = 5  # 임베딩 된 차원의 크기 및 RNN 층 입력 차원의 크기
hidden_size = 20  # RNN의 은닉층 크기

In [112]:
model = Net(vocab_size, input_size, hidden_size, batch_first=True)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [113]:
output = model(X)
print(output)

tensor([[ 0.2698, -0.0716, -0.2220, -0.0463,  0.0249,  0.4638, -0.1436,  0.0290],
        [ 0.3695,  0.0845,  0.2757, -0.1985,  0.0441,  0.1246, -0.1609,  0.2535],
        [ 0.2317,  0.0657, -0.2014, -0.2986,  0.1568,  0.2711, -0.1225,  0.2495],
        [-0.3509, -0.1471,  0.0847, -0.1859,  0.4180,  0.1520, -0.5403,  0.2543],
        [-0.3108,  0.3556, -0.1022, -0.2227, -0.2265,  0.1363,  0.0461,  0.1787],
        [-0.2641,  0.2909, -0.4598, -0.3608,  0.0418,  0.5764, -0.2460,  0.1149]],
       grad_fn=<ViewBackward0>)


In [114]:
output.shape

torch.Size([6, 8])

In [89]:
decode = lambda y: [index2word.get(x) for x in y]

In [90]:
decode(output.softmax(-1).argmax(-1).tolist())

['best', '<unk', 'memory', 'best', '<unk', 'best']

In [91]:
for step in range(201):
    # 경사 초기화
    optimizer.zero_grad()
    # 순방향 전파
    output = model(X)
    # 손실값 계산
    loss = loss_fn(output, Y.view(-1))
    # 역방향 전파
    loss.backward()
    # 매개변수 업데이트
    optimizer.step()
    # 기록
    if step % 40 == 0:
        print("[{:02d}/201] {:.4f} ".format(step+1, loss))
        pred = output.softmax(-1).argmax(-1).tolist()
        print(" ".join(["Repeat"] + decode(pred)))
        print()

[01/201] 2.0677 
Repeat best <unk memory best <unk best

[41/201] 1.4957 
Repeat is the best medicine for memory

[81/201] 0.8131 
Repeat is the best medicine for memory

[121/201] 0.3929 
Repeat is the best medicine for memory

[161/201] 0.2130 
Repeat is the best medicine for memory

[201/201] 0.1303 
Repeat is the best medicine for memory

